In [8]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('0.3_scaling_data.csv')

output_dir = '/Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets'
os.makedirs(output_dir, exist_ok=True)

burnout_map = {0: 'Low', 1: 'Moderate', 2: 'High'}
stress_map = {0: 'Low', 1: 'Moderate', 2: 'High'}

if pd.api.types.is_numeric_dtype(df['Burnout_Risk']):
    df['Burnout_Risk_Label'] = df['Burnout_Risk'].map(burnout_map)
else:
    df['Burnout_Risk_Label'] = df['Burnout_Risk']

if pd.api.types.is_numeric_dtype(df['Stress_Level']):
    df['Stress_Level_Label'] = df['Stress_Level'].map(stress_map)
else:
    df['Stress_Level_Label'] = df['Stress_Level']

order_risk = ['Low', 'Moderate', 'High']
palette_risk = {'Low': '#55A868', 'Moderate': '#F1A340', 'High': '#C44E52'}

# KHỞI TẠO DASHBOARD (Khung 2x2)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.set_theme(style="whitegrid")

# H1: Work Hours vs Burnout Risk (Boxplot)
sns.boxplot(data=df, x='Burnout_Risk_Label', y='Work_Hours_Per_Week', 
            order=order_risk, palette=palette_risk, ax=axes[0, 0])
axes[0, 0].set_title('H1: Work Hours Impact on Burnout Risk', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Burnout Risk', fontweight='bold')
axes[0, 0].set_ylabel('Work Hours Per Week')

# H2: Sleep Hours vs Burnout Risk (Violinplot)
sns.violinplot(data=df, x='Burnout_Risk_Label', y='Sleep_Hours', 
               order=order_risk, palette=palette_risk, inner="quartile", ax=axes[0, 1])
axes[0, 1].set_title('H2: Sleep Duration vs Burnout Risk', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Burnout Risk', fontweight='bold')
axes[0, 1].set_ylabel('Sleep Hours')

# H3: Screen Time vs Stress Level (Barplot)
sns.barplot(data=df, x='Stress_Level_Label', y='Screen_Time_Hours', 
            order=order_risk, palette='Blues', capsize=.1, ax=axes[1, 0])
axes[1, 0].set_title('H3: Screen Time contribution to Stress', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Stress Level', fontweight='bold')
axes[1, 0].set_ylabel('Average Screen Time (Hours)')

# H4: Lifestyle factors (Scatterplot) 
sns.scatterplot(data=df, x='Physical_Activity_Hours', y='Meditation_Minutes', 
                hue='Burnout_Risk_Label', hue_order=order_risk, palette=palette_risk, 
                alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title('H4: Lifestyle Mitigation (Activity & Meditation)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Physical Activity (Hours)')
axes[1, 1].set_ylabel('Meditation (Minutes)')
axes[1, 1].legend(title='Burnout Risk')

# TINH CHỈNH VÀ LƯU DASHBOARD
plt.suptitle('Burnout Risk Prediction: Key Hypotheses Dashboard', fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()

dash_img_name = '0.4_hypothesis_dashboard.png'
dash_img_path = os.path.join(output_dir, dash_img_name)
plt.savefig(dash_img_path, dpi=300, bbox_inches='tight')
print(f"![Hypothesis Dashboard]({dash_img_path})")
plt.close()

![Hypothesis Dashboard](/Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets/0.4_hypothesis_dashboard.png)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

df = pd.read_csv('0.4_scaled_data.csv')
X = df.drop(columns=['Burnout_Risk'])

# 2. Load the best tuned model
model_path = 'after_tuning.pkl'
if os.path.exists(model_path):
    best_model = joblib.load(model_path)
    print("Successfully loaded the 'after_tuning.pkl' model.")
else:
    raise FileNotFoundError(f"File '{model_path}' not found. Please ensure the tuning step is completed.")

#Extract Feature Importances
importances = best_model.feature_importances_
feature_names = X.columns

#Create a DataFrame for easy sorting
feature_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n TOP 5 MOST INFLUENTIAL FACTORS FOR BURNOUT ")
print(feature_imp_df.head(5).to_string(index=False))

#Visualize with a Horizontal Bar Chart
plt.figure(figsize=(10, 8))
sns.set_theme(style="whitegrid")

sns.barplot(
    x='Importance', 
    y='Feature', 
    data=feature_imp_df, 
    palette='viridis'
)

plt.title('Feature Importances in Predicting Burnout Risk', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Relative Importance (Contribution to model)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()

# Save the plot
output_dir = '/Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets'
os.makedirs(output_dir, exist_ok=True)
plot_filename = os.path.join(output_dir, 'feature_importance_rf.png')

plt.savefig(plot_filename, dpi=300)
plt.close()

print(f"\nFeature Importance plot saved successfully at: {plot_filename}")

Successfully loaded the 'after_tuning.pkl' model.

=== TOP 5 MOST INFLUENTIAL FACTORS FOR BURNOUT ===
                   Feature  Importance
              Stress_Level    0.429314
           Chronic_OHE_Yes    0.131248
            Chronic_OHE_No    0.110815
       Work_Hours_Per_Week    0.078016
Work_Hours_Per_Week_BoxCox    0.075140

Feature Importance plot saved successfully at: /Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets/feature_importance_rf.png
